## **Installs**

In [ ]:
#!pip install -U transformers

# **Common Sections**

## **Hugging Face Token**

In [1]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("my_hf_token")

In [2]:
from huggingface_hub import login
login(token=hf_token)

## **Imports**

In [3]:
import json
import re
import math
import random
import copy

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
import gc, traceback

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# **Part 1: Generating positive and negative responses**

## **Load Responses generating Model**

In [4]:
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
HF_TOKEN = hf_token

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    dtype=torch.float16,      # NOT torch_dtype (deprecated, as you already found)
    device_map="auto",
)
model.eval()

DEVICE = next(model.parameters()).device
NUM_LAYERS = model.config.num_hidden_layers
HIDDEN_DIM = model.config.hidden_size
print(f"Loaded {MODEL_NAME}: {NUM_LAYERS} layers, hidden_dim={HIDDEN_DIM}, device={DEVICE}")

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Loaded google/gemma-2b: 18 layers, hidden_dim=2048, device=cuda:0


In [6]:
#freeze model params

for p in model.parameters():
    p.requires_grad_(False)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params in base model (should be 0): {trainable}")

Trainable params in base model (should be 0): 0


## **Load Questions**

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/anarvaaa/dataset-for-ssv/pico_dataset_only_questions.csv")  # update path if running locally

print(df.shape)
df.head(3)

In [ ]:
GEN_MAX_NEW_TOKENS = 100
GEN_DO_SAMPLE = False
GEN_TEMPERATURE = 1.0
GEN_TOP_P = 0.95

## **Generate Positive Responses**

### **Example Positive Prompt:**

Answer the following clinical research question by extracting it into PICO format 
(Population, Intervention, Comparison, Outcome). Respond only with:
P: ...
I: ...
C: ...
O: ...

Question: In adults with out-of-hospital cardiac arrest (OHCA) and an initial 
shockable rhythm, does real-time audiovisual feedback during CPR compared with 
standard CPR without feedback improve survival to hospital discharge?



In [ ]:
# code to generate positive responces from the model

## **Generate Negative Responses**

### **Example Negative Prompt:**

In adults with out-of-hospital cardiac arrest (OHCA) and an initial shockable 
rhythm, does real-time audiovisual feedback during CPR compared with standard 
CPR without feedback improve survival to hospital discharge?

In [ ]:
# code to generate negative responses from the model

## **Save new dataset**

In [ ]:
# code which adds the positive repsonces and negative responses and create a new dataset -> pico_dataset_positive_negative.csv
# save this in kaggle/working
# before moving to part 2, this will be uploaded into the dataset entity so it can be used later

# **Part 2: Training Steering Vector**

## **Load Steering Vector training Model**

In [ ]:
MODEL_NAME = "google/gemma-2b"
HF_TOKEN = hf_token

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    dtype=torch.float16,      # NOT torch_dtype (deprecated, as you already found)
    device_map="auto",
)
model.eval()

DEVICE = next(model.parameters()).device
NUM_LAYERS = model.config.num_hidden_layers
HIDDEN_DIM = model.config.hidden_size
print(f"Loaded {MODEL_NAME}: {NUM_LAYERS} layers, hidden_dim={HIDDEN_DIM}, device={DEVICE}")

In [ ]:
#freeze model params

for p in model.parameters():
    p.requires_grad_(False)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params in base model (should be 0): {trainable}")

## **Loading Positive and Negative Dataset**

In [7]:
df = pd.read_csv("/kaggle/input/datasets/anarvaaa/dataset-for-ssv/pico_dataset_positive_negative.csv")  # update path if running locally
print(df.shape)
df.head(3)

(50, 8)


,srno,clinical_speciality,research_question,P,I,C,O,negative_response
0,1,Cardiology,In adults with heart failure with preserved ej...,Adults with HFpEF (LVEF ≥ 50%),"SGLT2 inhibitors (e.g., empagliflozin, dapagli...",Placebo + standard care,Composite of cardiovascular death and hospital...,"The study was a randomized, double-blind, plac..."
1,2,Cardiology,In patients undergoing percutaneous coronary i...,Adults undergoing PCI for acute coronary syndrome,Short-term DAPT (1–3 mo) followed by P2Y12 inh...,Standard DAPT for 12 months,Bleeding Academic Research Consortium (BARC) t...,"This is a randomized, double-blind, placebo-co..."
2,3,Cardiology,In adults with severe asymptomatic aortic sten...,Adults with severe asymptomatic aortic stenosis,Early elective AVR (surgical or transcatheter),Conservative watchful waiting,All-cause mortality; myocardial infarction at ...,The aim of this study was to evaluate the impa...


## **Train-Test Split and Prompt, Positive and Negative Triplet**

In [8]:
def format_positive(row):
    return (
        f"P: {row['P']}\n"
        f"I: {row['I']}\n"
        f"C: {row['C']}\n"
        f"O: {row['O']}"
    )

def format_prompt(row):
    return f"{row['research_question'].strip()}\n"

examples = []
for _, row in df.iterrows():
    examples.append({
        "prompt": format_prompt(row),
        "positive": format_positive(row) + tokenizer.eos_token,
        "negative": str(row["negative_response"]).strip() + tokenizer.eos_token,
    })

random.shuffle(examples)
n_val = max(5, int(0.1 * len(examples)))
val_examples = examples[:n_val]
train_examples = examples[n_val:]
print(f"train={len(train_examples)}  val={len(val_examples)}")
print("--- sample ---")
print("PROMPT:", train_examples[0]["prompt"])
print("POSITIVE:", train_examples[0]["positive"])
print("NEGATIVE:", train_examples[0]["negative"])

train=45  val=5
--- sample ---
PROMPT: In adults aged 45 and older with degenerative medial meniscus tears and no severe osteoarthritis, does arthroscopic partial meniscectomy compared with structured physical therapy improve patient-reported pain and function at 12 months?

POSITIVE: P: Adults aged ≥45 with symptomatic degenerative medial meniscus tears
I: Arthroscopic partial meniscectomy (APM)
C: Structured physical therapy/conservative exercise programs
O: Patient-reported knee pain and physical function (e.g., KOOS or WOMAC scores)<eos>
NEGATIVE: The purpose of this study was to compare the effectiveness of a 12-week structured physical therapy program with a 12-week arthroscopic partial meniscectomy program in patients with degenerative medial meniscus tears and no severe osteoarthritis.
The study was a randomized, controlled trial. Patients were randomly assigned to either the physical therapy group or the arthroscopic partial meniscectomy group. The primary outcome was pain mea

## **Steering Vector**

In [18]:
class SteeringVector(nn.Module):
    """Rank-1 steering vector.
    'add'  mode (eq. 9):  h <- h + alpha * w1 + b1
    'null' mode (eq. 7):  h <- h - (ReLU(h . w1) / ||w1||^2) * w1
    """
    def __init__(self, dim, dtype=torch.float32):
        super().__init__()
        self.w1 = nn.Parameter(torch.randn(dim, dtype=dtype) * 0.01)
        self.b1 = nn.Parameter(torch.zeros(dim, dtype=dtype))

    def forward(self, h, alpha=0.0, mode="add"):
        w1 = self.w1.to(device=h.device, dtype=h.dtype)
        if mode == "add":
            b1 = self.b1.to(device=h.device, dtype=h.dtype)
            return h + alpha * w1 + b1
        elif mode == "null":
            # project out the (ReLU-gated) component along w1, per eq. 7
            dot = torch.einsum("...d,d->...", h, w1)          # (..., )
            gated = F.relu(dot) / (w1.norm() ** 2 + 1e-8)      # (..., )
            return h - gated.unsqueeze(-1) * w1
        else:
            raise ValueError(mode)


class SteeringHook:
    """Forward hook that injects a SteeringVector into a decoder layer's output."""
    def __init__(self, steering_vector: SteeringVector):
        self.sv = steering_vector
        self.enabled = False
        self.alpha = 0.0
        self.mode = "add"     # "add" for Phi_Steer, "null" for Phi_Null
        self.handle = None

    def __call__(self, module, inputs, output):
        if not self.enabled:
            return output
        if isinstance(output, tuple):
            hidden_states = self.sv(output[0], self.alpha, self.mode)
            return (hidden_states,) + output[1:]
        else:
            return self.sv(output, self.alpha, self.mode)

    def attach(self, layer_module):
        self.handle = layer_module.register_forward_hook(self)
        return self.handle

    def remove(self):
        if self.handle is not None:
            self.handle.remove()
            self.handle = None

In [19]:
STEER_LAYER = max(1, int(NUM_LAYERS * 0.4))
layer_device = next(model.model.layers[STEER_LAYER].parameters()).device
print(f"Hooking layer index {STEER_LAYER} / {NUM_LAYERS} (on {layer_device})")

steering_vector = SteeringVector(HIDDEN_DIM, dtype=torch.float32).to(layer_device)
steering_hook = SteeringHook(steering_vector)
steering_hook.attach(model.model.layers[STEER_LAYER])

n_trainable = sum(p.numel() for p in steering_vector.parameters() if p.requires_grad)
print(f"Trainable steering-vector params: {n_trainable}  (2 x {HIDDEN_DIM})")

Hooking layer index 7 / 18 (on cuda:1)
Trainable steering-vector params: 4096  (2 x 2048)


## **Preference Objective**

In [20]:
def seq_logprob(prompt_ids, response_ids, alpha, device, mode="add", use_hook=True):
    """Length-normalized log p(response | prompt).
    use_hook=False -> reference (unsteered) model likelihood.
    """
    input_ids = torch.cat([prompt_ids, response_ids], dim=0).unsqueeze(0).to(device)
    attention_mask = torch.ones_like(input_ids)

    steering_hook.enabled = use_hook
    steering_hook.alpha = alpha
    steering_hook.mode = mode
    out = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = out.logits

    shift_logits = logits[:, :-1, :].float()
    shift_labels = input_ids[:, 1:]
    log_probs = F.log_softmax(shift_logits, dim=-1)
    token_logp = log_probs.gather(2, shift_labels.unsqueeze(-1)).squeeze(-1)

    P = prompt_ids.shape[0]
    R = response_ids.shape[0]
    resp_logp = token_logp[:, P - 1 : P - 1 + R]
    steering_hook.enabled = False   # always leave disabled after use
    return resp_logp.sum() / R


def build_pair(example, device):
    prompt_ids = tokenizer(example["prompt"], return_tensors="pt", add_special_tokens=True).input_ids[0]
    pos_ids = tokenizer(example["positive"], return_tensors="pt", add_special_tokens=False).input_ids[0]
    neg_ids = tokenizer(example["negative"], return_tensors="pt", add_special_tokens=False).input_ids[0]
    return prompt_ids, pos_ids, neg_ids

## **Training**

In [ ]:
TRAIN_FACTORS = [2.0, 4.0, 6.0, 8.0, 10.0, 12.0, 14.0, 16.0]
LR = 0.04
EPOCHS = 15
GRAD_CLIP = 1.0
WEIGHT_DECAY = 0.00
PATIENCE = 2

optimizer = torch.optim.AdamW(steering_vector.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

@torch.no_grad()
def precompute_reference_logprobs(examples, device):
    """log p(y|x) and log p(y^c|x) under the frozen, unsteered reference model.
    Cached once since the base model never changes (reference-free RePS only
    uses these to derive the beta weighting terms, not as a KL anchor)."""
    cache = {}
    for i, ex in enumerate(examples):
        prompt_ids, pos_ids, neg_ids = build_pair(ex, device)
        logp_pos_ref = seq_logprob(prompt_ids, pos_ids, alpha=0.0, device=device, use_hook=False)
        logp_neg_ref = seq_logprob(prompt_ids, neg_ids, alpha=0.0, device=device, use_hook=False)
        cache[i] = (logp_pos_ref.item(), logp_neg_ref.item())
    return cache

def reps_loss(example, ref_logps, alpha, device):
    prompt_ids, pos_ids, neg_ids = build_pair(example, device)
    logp_pos_ref, logp_neg_ref = ref_logps

    # beta weights from the reference model (eq. 5 / eq. 6)
    beta_pos = max(logp_neg_ref - logp_pos_ref, 1.0)   # weights Delta+
    beta_neg = max(logp_pos_ref - logp_neg_ref, 1.0)   # weights Delta-

    # Delta+ : winning=positive(steered), losing=negative(original), under Phi_Steer (eq. 5)
    logp_pos_steer = seq_logprob(prompt_ids, pos_ids, alpha, device, mode="add", use_hook=True)
    logp_neg_steer = seq_logprob(prompt_ids, neg_ids, alpha, device, mode="add", use_hook=True)
    delta_pos = beta_pos * logp_pos_steer - logp_neg_steer

    # Delta- : winning=negative(original), losing=positive(steered), under Phi_Null (eq. 6)
    logp_neg_null = seq_logprob(prompt_ids, neg_ids, alpha=0.0, device=device, mode="null", use_hook=True)
    logp_pos_null = seq_logprob(prompt_ids, pos_ids, alpha=0.0, device=device, mode="null", use_hook=True)
    delta_neg = beta_neg * logp_neg_null - logp_pos_null

    loss = -F.logsigmoid(delta_pos) - F.logsigmoid(delta_neg)
    return loss, delta_pos.item(), delta_neg.item()

@torch.no_grad()
def evaluate(examples, ref_cache, alpha, device):
    losses, margins = [], []
    for i, ex in enumerate(examples):
        loss, d_pos, d_neg = reps_loss(ex, ref_cache[i], alpha, device)
        losses.append(loss.item())
        margins.append(d_pos)  # track steering margin as before
    acc = float(np.mean([m > 0 for m in margins]))
    return float(np.mean(losses)), float(np.mean(margins)), acc


In [ ]:
print("Precomputing reference (unsteered) log-probs...")
train_ref_cache = precompute_reference_logprobs(train_examples, layer_device)
val_ref_cache = precompute_reference_logprobs(val_examples, layer_device)

history = {"train_loss": [], "val_loss": [], "val_margin": [], "val_acc": [], "vec_norm": []}
best_val_loss = float("inf")
best_state = None
epochs_since_improve = 0

for epoch in range(EPOCHS):
    idx = list(range(len(train_examples)))
    random.shuffle(idx)
    epoch_losses = []
    for i in idx:
        ex = train_examples[i]
        alpha = random.choice(TRAIN_FACTORS)

        optimizer.zero_grad()
        loss, _, _ = reps_loss(ex, train_ref_cache[i], alpha, layer_device)

        if not torch.isfinite(loss):
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(steering_vector.parameters(), GRAD_CLIP)
        optimizer.step()
        epoch_losses.append(loss.item())

    train_loss = float(np.mean(epoch_losses)) if epoch_losses else float("nan")
    val_loss, val_margin, val_acc = evaluate(val_examples, val_ref_cache,
                                              alpha=TRAIN_FACTORS[len(TRAIN_FACTORS)//2],
                                              device=layer_device)
    vec_norm = steering_vector.w1.norm().item()
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_margin"].append(val_margin)
    history["val_acc"].append(val_acc)
    history["vec_norm"].append(vec_norm)
    print(f"epoch {epoch+1:02d}/{EPOCHS}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  "
          f"val_margin={val_margin:+.4f}  val_pref_acc={val_acc:.2f}  ||w1||={vec_norm:.3f}")

    if val_loss < best_val_loss - 1e-4:
        best_val_loss = val_loss
        best_state = copy.deepcopy(steering_vector.state_dict())
        epochs_since_improve = 0
    else:
        epochs_since_improve += 1
        if epochs_since_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1}.")
            break

if best_state is not None:
    steering_vector.load_state_dict(best_state)
    print(f"Restored best checkpoint (val_loss={best_val_loss:.4f}, ||w1||={steering_vector.w1.norm().item():.3f})")

Precomputing reference (unsteered) log-probs...
epoch 01/15  train_loss=1.4847  val_loss=0.0151  val_margin=+9.7336  val_pref_acc=1.00  ||w1||=12.637
epoch 02/15  train_loss=0.6452  val_loss=1.7890  val_margin=+1.7604  val_pref_acc=0.60  ||w1||=14.802
epoch 03/15  train_loss=0.7308  val_loss=1.4078  val_margin=+0.2943  val_pref_acc=0.60  ||w1||=17.105
Early stopping at epoch 3.
Restored best checkpoint (val_loss=0.0151, ||w1||=12.637)


## **Best Steering Vector**

In [23]:
candidate_factors = [0.25, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0, 8.0]
factor_stats = {}
for f in candidate_factors:
    val_loss, val_margin, val_acc = evaluate(val_examples, val_ref_cache, alpha=f, device=layer_device)
    factor_stats[f] = {"loss": val_loss, "margin": val_margin, "acc": val_acc}
    print(f"alpha={f:>5.2f}  val_loss={val_loss:.4f}  val_margin={val_margin:+.4f}  val_pref_acc={val_acc:.2f}")

ACC_THRESHOLD = 0.8
qualifying = [f for f in candidate_factors if factor_stats[f]["acc"] >= ACC_THRESHOLD]
if qualifying:
    BEST_ALPHA = min(qualifying)
else:
    BEST_ALPHA = max(candidate_factors, key=lambda f: factor_stats[f]["acc"])
print(f"\nBest steering factor: {BEST_ALPHA}  "
      f"(smallest factor with val_pref_acc >= {ACC_THRESHOLD}, or best available)")
print("Sanity-check this with actual generations below -- log-prob margin alone")
print("does not guarantee fluent output; always eyeball a few samples.")

alpha= 0.25  val_loss=1.7334  val_margin=-1.4100  val_pref_acc=0.00
alpha= 0.50  val_loss=1.6615  val_margin=-1.2765  val_pref_acc=0.40
alpha= 1.00  val_loss=1.3822  val_margin=-0.6372  val_pref_acc=0.60
alpha= 1.50  val_loss=0.9366  val_margin=+0.8154  val_pref_acc=0.60
alpha= 2.00  val_loss=0.3717  val_margin=+3.1977  val_pref_acc=0.80
alpha= 3.00  val_loss=0.0081  val_margin=+9.6055  val_pref_acc=1.00
alpha= 4.00  val_loss=0.0018  val_margin=+14.7265  val_pref_acc=1.00
alpha= 6.00  val_loss=0.0018  val_margin=+16.8590  val_pref_acc=1.00
alpha= 8.00  val_loss=0.0019  val_margin=+13.7697  val_pref_acc=1.00

Best steering factor: 2.0  (smallest factor with val_pref_acc >= 0.8, or best available)
Sanity-check this with actual generations below -- log-prob margin alone
does not guarantee fluent output; always eyeball a few samples.


## **Generation**

In [24]:
@torch.no_grad()
def generate(prompt, alpha=None, max_new_tokens=100):
    steering_hook.enabled = alpha is not None
    steering_hook.alpha = alpha if alpha is not None else 0.0

    inputs = tokenizer(prompt, return_tensors="pt").to(layer_device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    text = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    steering_hook.enabled = False
    return text

In [25]:
for ex in val_examples[:3]:
    print("=" * 80)
    print("PROMPT:", ex["prompt"].strip())
    print("-" * 80)
    print("UNSTEERED:")
    print(generate(ex["prompt"], alpha=None))
    print("-" * 80)
    print(f"STEERED (learned vector, alpha={BEST_ALPHA}):")
    print(generate(ex["prompt"], alpha=BEST_ALPHA))
    print("-" * 80)
    print("POSITIVE (ground truth):")
    print(ex["positive"].replace(tokenizer.eos_token, ""))

PROMPT: In adults with out-of-hospital cardiac arrest (OHCA) and an initial shockable rhythm, does real-time audiovisual feedback during CPR compared with standard CPR without feedback improve survival to hospital discharge?
--------------------------------------------------------------------------------
UNSTEERED:
The study was a randomized controlled trial conducted in 10 hospitals in the United States. The study included 1,000 patients with OHCA and an initial shockable rhythm. Patients were randomly assigned to receive either real-time audiovisual feedback during CPR or standard CPR without feedback. The primary outcome was survival to hospital discharge.
The study found that patients who received real-time audiovisual feedback during CPR had a 10% higher survival to hospital discharge compared with patients who received standard CPR
--------------------------------------------------------------------------------
STEERED (learned vector, alpha=2.0):
The primary outcome of this stud

In [ ]:
torch.save(
    {
        "w1": steering_vector.w1.detach().cpu(),
        "b1": steering_vector.b1.detach().cpu(),
        "layer": STEER_LAYER,
        "best_alpha": BEST_ALPHA,
        "model_name": MODEL_NAME,
        "objective": "preference (positive vs negative)",
    },
    "/kaggle/working/pico_steering_vector_exp2.pt",
)
print("Saved /kaggle/working/pico_steering_vector_exp2.pt")